In [2]:
import cv2
import numpy as np
import random
import os
import json
from pycocotools import mask as mask_util

def create_random_convex_polygon(img_size, min_vertices=3, max_vertices=6):
    """创建随机凸多边形"""
    def polar_angle(point, center):
        """计算极角"""
        return np.arctan2(point[1] - center[1], point[0] - center[0])
    
    def distance(p1, p2):
        """计算两点距离"""
        return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

    # 生成足够多的随机点
    num_points = random.randint(min_vertices, max_vertices)
    center_x = random.randint(200, img_size[1]-200)
    center_y = random.randint(200, img_size[0]-200)
    center = np.array([center_x, center_y])
    
    points = []
    for _ in range(num_points):
        angle = random.uniform(0, 2 * np.pi)
        radius = random.randint(20, 50)
        x = int(center_x + radius * np.cos(angle))
        y = int(center_y + radius * np.sin(angle))
        points.append([x, y])
    
    # 按照极角排序
    points.sort(key=lambda p: polar_angle(p, center))
    
    # Graham扫描法构建凸包
    stack = []
    
    # 添加前两个点
    stack.append(points[0])
    stack.append(points[1])
    
    # 处理剩余的点
    for i in range(2, len(points)):
        while len(stack) > 1:
            p1 = np.array(stack[-2])
            p2 = np.array(stack[-1])
            p3 = np.array(points[i])
            
            # 计算叉积，判断是否是左转
            cross_product = (p2[0] - p1[0]) * (p3[1] - p1[1]) - (p2[1] - p1[1]) * (p3[0] - p1[0])
            
            if cross_product > 0:
                break
            stack.pop()
            
        stack.append(points[i])
    
    # 确保多边形闭合
    if len(stack) >= 3:
        return np.array(stack)
    else:
        # 如果点不够构成多边形，重新生成
        return create_random_convex_polygon(img_size, min_vertices, max_vertices)

def create_random_ellipse():
    """创建随机椭圆参数"""
    center = (random.randint(100, 924), random.randint(100, 924))
    axes = (random.randint(20, 50), random.randint(20, 50))
    angle = random.uniform(0, 360)
    return center, axes, angle

def check_overlap(mask, new_shape):
    """检查是否有重叠"""
    return np.any(mask & new_shape)

def binary_mask_to_rle(binary_mask):
    """将二值mask转换为RLE编码"""
    rle = mask_util.encode(np.asfortranarray(binary_mask))
    rle['counts'] = rle['counts'].decode('utf-8')  # 将bytes转换为string
    return rle

In [7]:
from sklearn.model_selection import train_test_split

def generate_dataset(num_images, output_dir):
    """生成数据集并划分训练集和测试集"""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if not os.path.exists(os.path.join(output_dir, 'images')):
        os.makedirs(os.path.join(output_dir, 'images'))

    img_size = (1024, 1024)
    all_data = {
        'images': [],
        'annotations': []
    }
    
    annotation_id = 0
    
    # 生成所有图像和标注
    for img_idx in range(num_images):
        # 创建白色背景
        image = np.ones((img_size[0], img_size[1], 3), dtype=np.uint8) * 255
        # 用于检查重叠的总mask
        total_mask = np.zeros(img_size, dtype=np.uint8)
        
        # 图像信息
        image_info = {
            'id': img_idx,
            'file_name': f'image_{img_idx}.png',
            'width': img_size[1],
            'height': img_size[0]
        }
        all_data['images'].append(image_info)
        
        # 随机生成2-5个形状
        num_shapes = random.randint(2, 5)
        
        for shape_idx in range(num_shapes):
            shape_type = random.choice(['polygon', 'ellipse'])
            mask = np.zeros(img_size, dtype=np.uint8)
            
            # 随机颜色
            color = (random.randint(0, 255), 
                    random.randint(0, 255), 
                    random.randint(0, 255))
            
            max_attempts = 50
            attempt = 0
            shape_created = False
            
            while attempt < max_attempts and not shape_created:
                temp_mask = np.zeros(img_size, dtype=np.uint8)
                
                if shape_type == 'polygon':
                    points = create_random_convex_polygon(img_size)
                    cv2.fillPoly(temp_mask, [points], 1)
                else:  # ellipse
                    center, axes, angle = create_random_ellipse()
                    cv2.ellipse(temp_mask, center, axes, angle, 0, 360, 1, -1)
                
                # 检查是否重叠
                if not check_overlap(total_mask, temp_mask):
                    mask = temp_mask
                    # 在图像上绘制形状
                    if shape_type == 'polygon':
                        cv2.fillPoly(image, [points], color)
                    else:
                        cv2.ellipse(image, center, axes, angle, 0, 360, color, -1)
                    shape_created = True
                    total_mask = total_mask | mask
                    
                    # 计算边界框
                    y_indices, x_indices = np.where(mask > 0)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x_min, x_max = np.min(x_indices), np.max(x_indices)
                        y_min, y_max = np.min(y_indices), np.max(y_indices)
                        bbox = [int(x_min), int(y_min), 
                               int(x_max - x_min), int(y_max - y_min)]
                        
                        # 创建标注信息
                        annotation = {
                            'id': annotation_id,
                            'image_id': img_idx,
                            'category_id': 1,  # 可以根据需要设置类别
                            'bbox': bbox,
                            'area': int(np.sum(mask)),
                            'segmentation': binary_mask_to_rle(mask),
                            'shape_type': shape_type
                        }
                        all_data['annotations'].append(annotation)
                        annotation_id += 1
                
                attempt += 1
        
        # 保存图像
        cv2.imwrite(os.path.join(output_dir, 'images', f'image_{img_idx}.png'), image)
    
    # 划分训练集和测试集
    image_ids = [img['id'] for img in all_data['images']]
    train_ids, test_ids = train_test_split(image_ids, test_size=0.1, random_state=42)

    # 创建训练集和测试集目录
    train_dir = os.path.join(output_dir, 'train')
    test_dir = os.path.join(output_dir, 'test')
    for d in [train_dir, test_dir]:
        if not os.path.exists(d):
            os.makedirs(os.path.join(d, 'images'))

    # 准备训练集和测试集的数据
    train_data = {'images': [], 'annotations': []}
    test_data = {'images': [], 'annotations': []}

    # 分配图像和标注
    for img in all_data['images']:
        if img['id'] in train_ids:
            train_data['images'].append(img)
            # 移动图像到训练集目录
            src = os.path.join(output_dir, 'images', img['file_name'])
            dst = os.path.join(train_dir, 'images', img['file_name'])
            os.rename(src, dst)
        else:
            test_data['images'].append(img)
            # 移动图像到测试集目录
            src = os.path.join(output_dir, 'images', img['file_name'])
            dst = os.path.join(test_dir, 'images', img['file_name'])
            os.rename(src, dst)

    for ann in all_data['annotations']:
        if ann['image_id'] in train_ids:
            train_data['annotations'].append(ann)
        else:
            test_data['annotations'].append(ann)

    # 删除原始images目录
    os.rmdir(os.path.join(output_dir, 'images'))

    # 保存训练集和测试集的标注文件
    with open(os.path.join(train_dir, 'annotations.json'), 'w') as f:
        json.dump(train_data, f)
    with open(os.path.join(test_dir, 'annotations.json'), 'w') as f:
        json.dump(test_data, f)

    # 打印数据集统计信息
    print(f"Dataset generated successfully!")
    print(f"Total images: {num_images}")
    print(f"Training images: {len(train_ids)}")
    print(f"Testing images: {len(test_ids)}")
    print(f"Total annotations: {len(all_data['annotations'])}")
    print(f"Training annotations: {len(train_data['annotations'])}")
    print(f"Testing annotations: {len(test_data['annotations'])}")

# 使用示例
output_dir = './'
num_images = 200
generate_dataset(num_images, output_dir)

Dataset generated successfully!
Total images: 200
Training images: 180
Testing images: 20
Total annotations: 715
Training annotations: 644
Testing annotations: 71
